# DAG Evaluation

## Single Graph Calculation

### Quantitative Metrics

#### Exploration Density

Exploration Density is defined as: **actual edges / maximum possible edges**

For Directed Acyclic Graphs (DAG):
- Actual edges: number of edges in the graph
- Maximum possible edges: For a DAG with n nodes, maximum edges occur when nodes are in linear topological order, where any earlier node can point to any later node, i.e., **n(n-1)/2**

Formula:
$$\text{Exploration Density} = \frac{|E|}{\frac{n(n-1)}{2}} = \frac{2|E|}{n(n-1)}$$

Where |E| is the actual number of edges, n is the number of nodes.

In [ ]:
def exploration_density(g: dict) -> float:
    n = len(g["nodes"])
    e = len(g["edges"])
    
    if n <= 1:
        return 0.0
    
    max_edges = n * (n - 1) / 2
    return e / max_edges

#### Branching Factor

Define **branching factor** as the **average excess branching** of all node out-degrees:

$$B = \frac{1}{n} \sum_{i=1}^{n} \max(0, d_i^{out} - 1)$$

In [ ]:
def branching_factor(g: dict) -> float:
    out_degree = {node["id"]: 0 for node in g["nodes"]}
    for edge in g["edges"]:
        out_degree[edge["from"]] += 1
    
    n = len(out_degree)
    if n == 0:
        return 0.0
    
    return sum(max(0, d - 1) for d in out_degree.values()) / n

#### Convergence Factor

**Convergence factor** measures the degree to which nodes are converged by multiple parent nodes:

$$C = \frac{1}{n} \sum_{i=1}^{n} \max(0, d_i^{in} - 1)$$

In [ ]:
def convergence_factor(g: dict) -> float:
    in_degree = {node["id"]: 0 for node in g["nodes"]}
    for edge in g["edges"]:
        in_degree[edge["to"]] += 1
    
    n = len(in_degree)
    if n == 0:
        return 0.0
    
    return sum(max(0, d - 1) for d in in_degree.values()) / n

#### Linearity

**Linearity** measures the prevalence of strictly sequential reasoning in the graph.

Definition: Proportion of nodes with degree ≤ 2 (degree = in-degree + out-degree)

$$\ell(G) = 1 - \frac{|\{s \in V \mid d(s) > 2\}|}{|V|}$$

- Pure chain structure: all nodes have degree ≤ 2, linearity = 1
- Highly branched/converged: most nodes have degree > 2, linearity approaches 0

In [ ]:
def linearity(g: dict) -> float:
    n = len(g["nodes"])
    if n == 0:
        return 0.0
    
    degree = {node["id"]: 0 for node in g["nodes"]}
    for edge in g["edges"]:
        degree[edge["from"]] += 1
        degree[edge["to"]] += 1
    
    high_degree_count = sum(1 for d in degree.values() if d > 2)
    return 1 - high_degree_count / n

#### Dangling Node Count

**Dangling nodes** are nodes with out-degree = 0 (leaf nodes).

Reason for subtracting 1: A normal DAG has at least one terminal node, and any beyond that are "dangling".

$$\text{Dangling Node Count} = |\{v \in V \mid d^{out}(v) = 0\}| - 1$$

In [ ]:
def dangling_count(g: dict) -> int:
    out_degree = {node["id"]: 0 for node in g["nodes"]}
    for edge in g["edges"]:
        out_degree[edge["from"]] += 1
    
    zero_out = sum(1 for d in out_degree.values() if d == 0)
    return zero_out - 1

### Qualitative Metrics

#### Fork Nodes

1. **Final Answer Node**: Node with out-degree = 0 and largest ID
2. **Dangling Node**: Node with out-degree = 0 but not the final answer node
3. **Fork Point**: The **Lowest Common Ancestor** (LCA) of dangling nodes and final answer node, i.e., the node where two paths diverge

Finding method:
- For each dangling node, trace back all its ancestor nodes
- For the final answer node, trace back all its ancestor nodes
- Find the closest common ancestor, which is the fork point

When two dangling nodes share a fork point, the fork node will be output repeatedly. Use the dedup parameter to remove duplicates.

In [ ]:
def find_fork_nodes(g: dict, dedup: bool = False) -> list:
    nodes = {node["id"]: node for node in g["nodes"]}
    
    # Calculate out-degree
    out_degree = {nid: 0 for nid in nodes}
    for edge in g["edges"]:
        out_degree[edge["from"]] += 1
    
    # Build parent node mapping
    parents = {nid: [] for nid in nodes}
    for edge in g["edges"]:
        parents[edge["to"]].append(edge["from"])
    
    # Find nodes with out-degree=0, sorted by ID
    zero_out = [nid for nid, d in out_degree.items() if d == 0]
    zero_out.sort(key=lambda x: int(x[1:]))  # Assume ID format is H1, H2...
    
    final_node = zero_out[-1]
    dangling_nodes = zero_out[:-1]
    
    # Get all ancestors
    def get_ancestors(nid):
        ancestors = set()
        stack = [nid]
        while stack:
            curr = stack.pop()
            for p in parents[curr]:
                if p not in ancestors:
                    ancestors.add(p)
                    stack.append(p)
        return ancestors
    
    final_ancestors = get_ancestors(final_node) | {final_node}
    
    # Find fork points for each dangling node
    result = []
    for dn in dangling_nodes:
        # Trace up from dangling node to find first node in final_ancestors
        visited = set()
        stack = [dn]
        fork = None
        while stack and fork is None:
            curr = stack.pop()
            for p in parents[curr]:
                if p in final_ancestors:
                    fork = p
                    break
                if p not in visited:
                    visited.add(p)
                    stack.append(p)
        if fork:
            result.append(nodes[fork])
    
    if dedup:
        unique_result = {}
        for node in result:
            unique_result[node["id"]] = node
        result = list(unique_result.values())
    
    return result

### Read and Calculate

In [ ]:
import json

def calculate_line(g: dict) -> dict:
    try:
        fork_nodes = find_fork_nodes(g, dedup=True)
    except Exception as e:
        fork_nodes = []
        # print("Error in find_fork_nodes:", e)
    return {
        'exploration_density': exploration_density(g),
        'branching_factor': branching_factor(g),
        'convergence_factor': convergence_factor(g),
        'linearity': linearity(g),
        'dangling_count': dangling_count(g),
        'fork_nodes': fork_nodes
    }

def read_gt_graphs(gt_path: str):
    gt_graphs = {}
    with open(gt_path, 'r', encoding='utf-8') as infile:
        for line in infile:
            data = json.loads(line)
            uuid = data['uuid']
            gt_graphs[uuid] = {
                'nodes': data['nodes'],
                'edges': data['edges']
            }
            gt_graphs[uuid].update(calculate_line(gt_graphs[uuid]))
    print(f"{gt_path} read completed, containing {len(gt_graphs)} graphs")
    return gt_graphs

def read_pred_graphs(data_path: str):
    pred_graphs = {}
    with open(data_path, 'r', encoding='utf-8') as infile:
        for line in infile:
            data = json.loads(line)
            uuid = data['uuid']
            pred_graphs[uuid] = {
                'nodes': data['nodes'],
                'edges': data['edges'],
                'matches': data['matches']
            }
            try:
                pred_graphs[uuid].update(calculate_line(pred_graphs[uuid]))
            except Exception as e:
                print(f"Error in calculate_line for {uuid}: {e}")
    print(f"{data_path} read completed, containing {len(pred_graphs)} graphs")
    if len(pred_graphs) < 500:
        print("less than 500 due to failure of LLM answer")
    return pred_graphs

### Visualization

In [ ]:
gt_path = './cleaned/ground_truth_graphs_cleaned.jsonl'
data_paths = [
    './cleaned/match_results_false__gpt-5_high__v5_merged.jsonl',
    './cleaned/match_results_true__gpt-5_high__v5_merged.jsonl',
    './cleaned/match_results_false__gpt-5_medium__v5_merged.jsonl',
    './cleaned/match_results_true__gpt-5_medium__v5_merged.jsonl',
    './cleaned/match_results_false__gpt-5_low__v5_merged.jsonl',
    './cleaned/match_results_true__gpt-5_low__v5_merged.jsonl',
    './cleaned/match_results_false__gpt-5_minimal__v5_merged.jsonl',
    './cleaned/match_results_true__gpt-5_minimal__v5_merged.jsonl',
    './cleaned/match_results_false__o3-2025-04-16_high__v5_merged.jsonl',
    './cleaned/match_results_true__o3-2025-04-16_high__v5_merged.jsonl',
    './cleaned/match_results_false__gpt-4_1__v5_merged.jsonl',
    './cleaned/match_results_true__gpt-4_1__v5_merged.jsonl',
    './cleaned/match_results_false__gpt-4o-2024-11-20__v5_merged.jsonl',
    './cleaned/match_results_true__gpt-4o-2024-11-20__v5_merged.jsonl',
    './cleaned/match_results_false__gemini-2_5-pro_high__v5_merged.jsonl',
    './cleaned/match_results_true__gemini-2_5-pro_high__v5_merged.jsonl',
    './cleaned/match_results_false__deepseek-reasoner-v32-special__v5_merged.jsonl',
    './cleaned/match_results_false__deepseek-v3_1-think-128k__v5_merged.jsonl',
    './cleaned/match_results_false__deepseek-v3_1-128k__v5_merged.jsonl',
    './cleaned/match_results_false__doubao-seed-1-6-251015_high__v5_merged.jsonl',
    './cleaned/match_results_false__kimi-k2-thinking__v5_merged.jsonl',
    './cleaned/match_results_false__qwen3-235b-a22b-thinking-2507__v5_merged.jsonl',
    './cleaned/match_results_false__intern-s1__v5_merged.jsonl',
    './cleaned/match_results_false__Qwen2_5-14B-Instruct__v5_merged.jsonl',
    './cleaned/match_results_false__ChemDFM_R__v5_merged.jsonl',
    './cleaned/match_results_false__Mistral-Small-24B-Instruct__v5_merged.jsonl',
    './cleaned/match_results_false__ether0__v5_merged.jsonl',
]

gt_graphs = read_gt_graphs(gt_path)

gpt_5_high_false_graphs = read_pred_graphs(data_paths[0])
gpt_5_high_true_graphs = read_pred_graphs(data_paths[1])
gpt_5_medium_false_graphs = read_pred_graphs(data_paths[2])
gpt_5_medium_true_graphs = read_pred_graphs(data_paths[3])
gpt_5_low_false_graphs = read_pred_graphs(data_paths[4])
gpt_5_low_true_graphs = read_pred_graphs(data_paths[5])
gpt_5_minimal_false_graphs = read_pred_graphs(data_paths[6])
gpt_5_minimal_true_graphs = read_pred_graphs(data_paths[7])
o3_high_false_graphs = read_pred_graphs(data_paths[8])
o3_high_true_graphs = read_pred_graphs(data_paths[9])
gpt_4_1_false_graphs = read_pred_graphs(data_paths[10])
gpt_4_1_true_graphs = read_pred_graphs(data_paths[11])
gpt_4o_false_graphs = read_pred_graphs(data_paths[12])
gpt_4o_true_graphs = read_pred_graphs(data_paths[13])
gemini_2_5_pro_high_false_graphs = read_pred_graphs(data_paths[16])
gemini_2_5_pro_high_true_graphs = read_pred_graphs(data_paths[17])
deepseek_v3_1_think_128k_false_graphs = read_pred_graphs(data_paths[19])
deepseek_v3_1_128k_false_graphs = read_pred_graphs(data_paths[20])
doubao_seed_1_6_251015_high_false_graphs = read_pred_graphs(data_paths[21])
kimi_k2_thinking_false_graphs = read_pred_graphs(data_paths[22])
qwen3_235b_a22b_thinking_2507_false_graphs = read_pred_graphs(data_paths[23])
intern_s1_false_graphs = read_pred_graphs(data_paths[24])
qwen2_5_14b_instruct_false_graphs = read_pred_graphs(data_paths[25])
chemdfm_r_false_graphs = read_pred_graphs(data_paths[26])
mistral_small_24b_instruct_false_graphs = read_pred_graphs(data_paths[27])
ether0_false_graphs = read_pred_graphs(data_paths[28])

pred_graphs_list = [
    gpt_5_high_false_graphs,
    gpt_5_high_true_graphs,
    gpt_5_medium_false_graphs,
    gpt_5_medium_true_graphs,
    gpt_5_low_false_graphs,
    gpt_5_low_true_graphs,
    gpt_5_minimal_false_graphs,
    gpt_5_minimal_true_graphs,
    o3_high_false_graphs,
    o3_high_true_graphs,
    gpt_4_1_false_graphs,
    gpt_4_1_true_graphs,
    gpt_4o_false_graphs,
    gpt_4o_true_graphs,
    gemini_2_5_pro_high_false_graphs,
    gemini_2_5_pro_high_true_graphs,
    deepseek_v3_1_think_128k_false_graphs,
    deepseek_v3_1_128k_false_graphs,
    doubao_seed_1_6_251015_high_false_graphs,
    kimi_k2_thinking_false_graphs,
    qwen3_235b_a22b_thinking_2507_false_graphs,
    intern_s1_false_graphs,
    qwen2_5_14b_instruct_false_graphs,
    chemdfm_r_false_graphs,
    mistral_small_24b_instruct_false_graphs,
    ether0_false_graphs,
]


labels = [
    'GPT-5 High Text-only',
    'GPT-5 High Multimodal',
    'GPT-5 Medium Text-only',
    'GPT-5 Medium Multimodal',
    'GPT-5 Low Text-only',
    'GPT-5 Low Multimodal',
    'GPT-5 Minimal Text-only',
    'GPT-5 Minimal Multimodal',
    'o3 High Text-only',
    'o3 High Multimodal',
    'GPT-4_1 Text-only',
    'GPT-4_1 Multimodal',
    'GPT-4o Text-only',
    'GPT-4o Multimodal',
    'Gemini-2.5-Pro-High Text-only',
    'Gemini-2.5-Pro-High Multimodal',
    'DeepSeek-V3.1-Think-128k Text-only',
    'DeepSeek-V3.1-128k Text-only',
    'Doubao-Seed-1-6-251015-High Text-only',
    'Kimi-K2-Thinking Text-only',
    'Qwen3-235b-A22b-Thinking-2507 Text-only',
    'Intern-S1 Text-only',
    'Qwen2_5-14B-Instruct Text-only',
    'ChemDFM-R Text-only',
    'Mistral-Small-24B-Instruct Text-only',
    'Ether0 Text-only',
]

pass1_list = []
for data_path in data_paths:
    pass1_cnt = 0
    with open(data_path, 'r', encoding='utf-8') as infile:
        for line in infile:
            data = json.loads(line)
            if 'score' in data:
                if data['score'] == 1:
                    pass1_cnt += 1
            else:
                print(f"no score in {data_path}, uuid: {data['uuid']}")
    pass1_list.append(round(pass1_cnt / 500, 3))
print(len(pass1_list))

In [ ]:
import matplotlib.pyplot as plt
from typing import List, Dict, Optional
import numpy as np

def plot_metric_distribution(metric: str, gt_graphs: Optional[Dict], pred_graphs_list: Optional[List[Dict]], labels: List[str]):
    fig, ax = plt.subplots(figsize=(12, 8), dpi=150)
    
    # Prepare data
    data = []
    all_labels = []
    
    if gt_graphs is not None:
        data.append([g[metric] for g in gt_graphs.values()])
        all_labels.append('Ground Truth')
    
    if pred_graphs_list is not None:
        for pred_graphs, label in zip(pred_graphs_list, labels):
            data.append([g[metric] for g in pred_graphs.values()])
            all_labels.append(label)
    
    # Draw horizontal violin plot
    parts = ax.violinplot(data, vert=False, showmeans=True)
    
    ax.set_yticks(np.arange(1, len(all_labels) + 1))
    ax.set_yticklabels(all_labels)
    ax.set_xlabel(metric)
    ax.set_title(f'Distribution of {metric}')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Exploration Density
plot_metric_distribution(
    metric='exploration_density',
    gt_graphs=gt_graphs,
    pred_graphs_list=pred_graphs_list,
    labels=labels
)

In [ ]:
# Branching Factor
plot_metric_distribution(
    metric='branching_factor',
    gt_graphs=gt_graphs,
    pred_graphs_list=pred_graphs_list,
    labels=labels
)

In [ ]:
# Convergence Factor
plot_metric_distribution(
    metric='convergence_factor',
    gt_graphs=gt_graphs,
    pred_graphs_list=pred_graphs_list,
    labels=labels
)

In [ ]:
# Linearity
plot_metric_distribution(
    metric='linearity',
    gt_graphs=gt_graphs,
    pred_graphs_list=pred_graphs_list,
    labels=labels
)

In [ ]:
# Dangling Node Count
plot_metric_distribution(
    metric='dangling_count',
    gt_graphs=None,
    pred_graphs_list=pred_graphs_list,
    labels=labels
)

In [ ]:
# Calculate and visualize mean values for each metric across models
metrics = ['exploration_density', 'branching_factor', 'convergence_factor', 'linearity', 'dangling_count']
all_graphs = [gt_graphs] + pred_graphs_list
all_labels = ['Ground Truth'] + labels

means = {label: [np.mean([g[m] for g in graphs.values()]) for m in metrics] 
         for label, graphs in zip(all_labels, all_graphs)}
# print(means)

metrics.append('pass@1')
for label in all_labels:
    if label == 'Ground Truth':
        means[label].append(0)
    else:
        means[label].append(pass1_list[labels.index(label)])

x = np.arange(len(metrics))
width = 0.03
fig, ax = plt.subplots(figsize=(22, 7), dpi=150)

# Use tab20 and tab20b color schemes, supporting up to 40 colors
colors = plt.cm.tab20.colors + plt.cm.tab20b.colors + plt.cm.tab20c.colors

for i, (label, values) in enumerate(means.items()):
    color = colors[i % len(colors)]
    ax.bar(x + i * width, values, width, label=label, color=color)

ax.set_xlabel('Metrics')
ax.set_ylabel('Mean Value')
ax.set_title('Mean Values by Metric and Model')
ax.set_xticks(x + width * (len(all_labels) - 1) / 2)
ax.set_xticklabels(metrics)
ax.legend(bbox_to_anchor=(1.05, 1.), loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Ground Truth - LLM Graph Calculation

### Quantitative Metrics

#### Recall and Precision

| Metric | Name | Meaning |
|------|------|------|
| **RPF** | Reasoning Path Fidelity | Fidelity considering node coverage and path structure |
| **Recall** | Recall | Degree to which ground truth nodes are correctly covered |
| **Precision** | Precision | How many LLM nodes are valid |
| **F1** | F1 Score | Harmonic mean of recall and precision |


```
Input: gt_graph, llm_graph

1. Build graph structure
   ├── G_gt: Ground Truth DAG
   └── G_llm: LLM DAG → Compute transitive closure

2. Handle many-to-one mapping
   ├── Extract mapping from llm_graph["matches"]
   ├── gt_to_llm: dict[r_id → list[h_id]]  # One ground truth node may correspond to multiple LLM nodes
   └── llm_to_gt: dict[h_id → r_id]        # Each LLM node corresponds to at most one ground truth node

3. Calculate RPF (anchored on ground truth nodes)
   Iterate through each node in gt_graph:
   │
   ├── Unmatched → score = 0
   │
   └── Matched → calculate logic_ratio
       ├── Get all parent nodes of this node in G_gt
       ├── For each parent: check if any corresponding LLM node can reach any LLM node of current node
       └── logic_ratio = valid parent nodes / total parent nodes (1 if no parents)
       
       node_score = points × logic_ratio
   
   RPF = Σ(node_score) / Σ(points)

4. Calculate other metrics
   ├── Recall = Σ(node_score) / Σ(points)
   ├── Precision = matched_llm_nodes / total_llm_nodes
   └── F1 = 2 * (Recall * Precision) / (Recall + Precision)

5. Output
   {
     'rpf': float,         # 0-1
     'recall': float,      # 0-1
     'precision': float,   # 0-1
     'f1': float,          # 0-1
     'node_details': dict  # Match details for each ground truth node
   }
```


In [ ]:
import networkx as nx
from collections import defaultdict

def calculate_dag_similarity(gt_graph: dict, llm_graph: dict) -> dict:
    """
    Calculate structural similarity between LLM reasoning path and ground truth path
    
    Args:
        gt_graph: Ground truth graph {'nodes': [{'id', 'points'}], 'edges': [{'from', 'to'}]}
        llm_graph: LLM graph {'nodes': [{'id'}], 'edges': [{'from', 'to'}], 'matches': [{'h_id', 'r_id'}]}
    
    Returns:
        {
            'rpf': float,
            'node_recall': float, 'node_precision': float, 'node_f1': float,
            'edge_recall': float, 'edge_precision': float, 'edge_f1': float,
            'node_details': dict, 'edge_details': dict
        }
    """
    
    # 1. Build graph
    G_gt = nx.DiGraph()
    G_gt.add_nodes_from(n['id'] for n in gt_graph['nodes'])
    G_gt.add_edges_from((e['from'], e['to']) for e in gt_graph['edges'])
    gt_closure = nx.transitive_closure(G_gt)
    
    G_llm = nx.DiGraph()
    G_llm.add_nodes_from(n['id'] for n in llm_graph['nodes'])
    G_llm.add_edges_from((e['from'], e['to']) for e in llm_graph['edges'])
    llm_closure = nx.transitive_closure(G_llm)
    
    # 2. Build mapping (supporting many-to-one)
    gt_to_llm = defaultdict(list)
    llm_to_gt = {}
    for m in llm_graph.get('matches', []):
        if m['r_id'] is None or m['h_id'] is None:
            continue
        gt_to_llm[m['r_id']].append(m['h_id'])
        llm_to_gt[m['h_id']] = m['r_id']
    
    # 3. Calculate node metrics
    gt_nodes = {n['id']: n for n in gt_graph['nodes']}
    gt_points = {n['id']: n['points'] for n in gt_graph['nodes']}
    total_score = 0.0
    max_score = sum(gt_points.values())
    node_details = {}
    
    for r_id, points in gt_points.items():
        h_ids = gt_to_llm.get(r_id, [])
        
        # Get complete node information
        gt_node = gt_nodes[r_id].copy()
        
        if not h_ids:
            node_details[r_id] = {**gt_node, 'matched': False, 'logic_ratio': 0, 'score': 0}
            continue
        
        parents = list(G_gt.predecessors(r_id))
        if not parents:
            logic_ratio = 1.0
        else:
            valid = sum(
                1 for p_id in parents
                if any(llm_closure.has_edge(p_h, h) 
                       for p_h in gt_to_llm.get(p_id, []) 
                       for h in h_ids)
            )
            logic_ratio = valid / len(parents)
        
        score = points * logic_ratio
        total_score += score
        node_details[r_id] = {**gt_node, 'matched': True, 'logic_ratio': logic_ratio, 'score': score}
    
    rpf = total_score / max_score if max_score > 0 else -1
    
    # Node recall / precision / f1
    matched_gt_nodes = set(gt_to_llm.keys()) & set(G_gt.nodes())
    matched_llm_nodes = set(llm_to_gt.keys()) & set(G_llm.nodes())
    
    node_recall = len(matched_gt_nodes) / len(G_gt.nodes()) if G_gt.nodes() else -1
    node_precision = len(matched_llm_nodes) / len(G_llm.nodes()) if G_llm.nodes() else -1
    node_f1 = 2 * node_recall * node_precision / (node_recall + node_precision) if (node_recall + node_precision) > 0 else -1
    
    # 4. Calculate edge metrics
    edge_details = {'gt_edges': {}, 'llm_edges': {}}
    
    # Edge recall: ground truth edges preserved in LLM transitive closure
    gt_edge_hits = 0
    for e in gt_graph['edges']:
        from_llms = gt_to_llm.get(e['from'], [])
        to_llms = gt_to_llm.get(e['to'], [])
        hit = any(llm_closure.has_edge(f, t) for f in from_llms for t in to_llms)
        edge_details['gt_edges'][(e['from'], e['to'])] = hit
        if hit:
            gt_edge_hits += 1
    
    edge_recall = gt_edge_hits / len(gt_graph['edges']) if gt_graph['edges'] else -1
    
    # Edge precision: LLM edges have basis in ground truth transitive closure
    llm_edge_hits = 0
    for e in llm_graph['edges']:
        gt_from = llm_to_gt.get(e['from'])
        gt_to = llm_to_gt.get(e['to'])
        hit = gt_from is not None and gt_to is not None and gt_closure.has_edge(gt_from, gt_to)
        edge_details['llm_edges'][(e['from'], e['to'])] = hit
        if hit:
            llm_edge_hits += 1
    
    edge_precision = llm_edge_hits / len(llm_graph['edges']) if llm_graph['edges'] else -1
    edge_f1 = 2 * edge_recall * edge_precision / (edge_recall + edge_precision) if (edge_recall + edge_precision) > 0 else -1
    
    return {
        'rpf': rpf,
        'node_recall': node_recall,
        'node_precision': node_precision,
        'node_f1': node_f1,
        'edge_recall': edge_recall,
        'edge_precision': edge_precision,
        'edge_f1': edge_f1,
        'node_details': node_details,
        'edge_details': edge_details
    }

### Read and Calculate

In [ ]:
def calculate_similarities(gt_graphs: dict, pred_graphs: dict) -> dict:
    """Calculate similarity for each pair of graphs"""
    results = {}
    for uuid in gt_graphs:
        if uuid in pred_graphs:
            results[uuid] = calculate_dag_similarity(gt_graphs[uuid], pred_graphs[uuid])
    return results

# Calculate similarity for all models
gpt_5_high_false_sim = calculate_similarities(gt_graphs, gpt_5_high_false_graphs)
gpt_5_high_true_sim = calculate_similarities(gt_graphs, gpt_5_high_true_graphs)
gpt_5_medium_false_sim = calculate_similarities(gt_graphs, gpt_5_medium_false_graphs)
gpt_5_medium_true_sim = calculate_similarities(gt_graphs, gpt_5_medium_true_graphs)
gpt_5_low_false_sim = calculate_similarities(gt_graphs, gpt_5_low_false_graphs)
gpt_5_low_true_sim = calculate_similarities(gt_graphs, gpt_5_low_true_graphs)
gpt_5_minimal_false_sim = calculate_similarities(gt_graphs, gpt_5_minimal_false_graphs)
gpt_5_minimal_true_sim = calculate_similarities(gt_graphs, gpt_5_minimal_true_graphs)
o3_high_false_sim = calculate_similarities(gt_graphs, o3_high_false_graphs)
o3_high_true_sim = calculate_similarities(gt_graphs, o3_high_true_graphs)
gpt_4_1_false_sim = calculate_similarities(gt_graphs, gpt_4_1_false_graphs)
gpt_4_1_true_sim = calculate_similarities(gt_graphs, gpt_4_1_true_graphs)
gpt_4o_false_sim = calculate_similarities(gt_graphs, gpt_4o_false_graphs)
gpt_4o_true_sim = calculate_similarities(gt_graphs, gpt_4o_true_graphs)
gemini_2_5_pro_high_false_sim = calculate_similarities(gt_graphs, gemini_2_5_pro_high_false_graphs)
gemini_2_5_pro_high_true_sim = calculate_similarities(gt_graphs, gemini_2_5_pro_high_true_graphs)
deepseek_v3_1_think_128k_false_sim = calculate_similarities(gt_graphs, deepseek_v3_1_think_128k_false_graphs)
deepseek_v3_1_128k_false_sim = calculate_similarities(gt_graphs, deepseek_v3_1_128k_false_graphs)
doubao_seed_1_6_251015_high_false_sim = calculate_similarities(gt_graphs, doubao_seed_1_6_251015_high_false_graphs)
kimi_k2_thinking_false_sim = calculate_similarities(gt_graphs, kimi_k2_thinking_false_graphs)
qwen3_235b_a22b_thinking_2507_false_sim = calculate_similarities(gt_graphs, qwen3_235b_a22b_thinking_2507_false_graphs)
intern_s1_false_sim = calculate_similarities(gt_graphs, intern_s1_false_graphs)
qwen2_5_14b_instruct_false_sim = calculate_similarities(gt_graphs, qwen2_5_14b_instruct_false_graphs)
chemdfm_r_false_sim = calculate_similarities(gt_graphs, chemdfm_r_false_graphs)
mistral_small_24b_instruct_false_sim = calculate_similarities(gt_graphs, mistral_small_24b_instruct_false_graphs)
ether0_false_sim = calculate_similarities(gt_graphs, ether0_false_graphs)

sim_results = [
    gpt_5_high_false_sim,
    gpt_5_high_true_sim,
    gpt_5_medium_false_sim,
    gpt_5_medium_true_sim,
    gpt_5_low_false_sim,
    gpt_5_low_true_sim,
    gpt_5_minimal_false_sim,
    gpt_5_minimal_true_sim,
    o3_high_false_sim,
    o3_high_true_sim,
    gpt_4_1_false_sim,
    gpt_4_1_true_sim,
    gpt_4o_false_sim,
    gpt_4o_true_sim,
    gemini_2_5_pro_high_false_sim,
    gemini_2_5_pro_high_true_sim,
    deepseek_v3_1_think_128k_false_sim,
    deepseek_v3_1_128k_false_sim,
    doubao_seed_1_6_251015_high_false_sim,
    kimi_k2_thinking_false_sim,
    qwen3_235b_a22b_thinking_2507_false_sim,
    intern_s1_false_sim,
    qwen2_5_14b_instruct_false_sim,
    chemdfm_r_false_sim,
    mistral_small_24b_instruct_false_sim,
    ether0_false_sim,
]
print(f"Similarity calculation completed")

### Visualization

In [ ]:
def plot_similarity_distribution(metric: str, sim_results: List[dict], labels: List[str]):
    fig, ax = plt.subplots(figsize=(12, 8), dpi=150)
    
    # Prepare data
    data = []
    for sim_dict in sim_results:
        data.append([s[metric] for s in sim_dict.values()])
    
    # Draw vertical violin plot
    pparts = ax.violinplot(data, vert=False, showmeans=True)
    
    ax.set_yticks(np.arange(1, len(all_labels) + 1))
    ax.set_yticklabels(all_labels)
    ax.set_xlabel(metric)
    ax.set_title(f'Distribution of {metric}')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# RPF
plot_similarity_distribution(
    metric='rpf',
    sim_results=sim_results,
    labels=labels
)

In [ ]:
# Recall distribution
plot_similarity_distribution('node_recall', sim_results, labels)

In [ ]:
# Precision distribution
plot_similarity_distribution('node_precision', sim_results, labels)

In [ ]:
# Calculate and visualize average recall and precision for each model
metrics = ['rpf', 'node_recall', 'node_precision', 'edge_recall', 'edge_precision']
means = {label: [np.mean([s[m] for s in sim_dict.values()]) for m in metrics] 
         for label, sim_dict in zip(labels, sim_results)}

metrics.append('pass@1')
for label in labels:
    means[label].append(pass1_list[labels.index(label)])

x = np.arange(len(metrics))
width = 0.03
fig, ax = plt.subplots(figsize=(22, 7), dpi=150)

# Define color scheme based on model type and configuration
# GPT-5 series - Blue (High/Medium/Low/Minimal, Text-only/Multimodal)
# o3 series - Green
# GPT-4.1 series - Cyan
# GPT-4o series - Purple
# Gemini series - Orange
# DeepSeek series - Red
# Doubao series - Brown
# Kimi series - Pink
# Qwen series - Yellow
# Intern series - Gray
# ChemDFM-R series - Dark green
# Ether0 series - Dark purple

color_map = {

    'GPT-5 High Text-only': '#2E5F8C',           # Deep Steel Blue
    'GPT-5 High Multimodal': '#8FB8D8',          # Misty Blue
    'GPT-5 Medium Text-only': '#4A7C9B',         # Medium Steel
    'GPT-5 Medium Multimodal': '#A3C9E0',        # Sky Gray-Blue
    'GPT-5 Low Text-only': '#6B9AC4',            # Soft Cerulean
    'GPT-5 Low Multimodal': '#BFD9ED',           # Pale Blue
    'GPT-5 Minimal Text-only': '#8FAEC4',        # Gray-Blue
    'GPT-5 Minimal Multimodal': '#D4E6F0',       # Whitish Blue
    
    'o3 High Text-only': '#2D6A5E',              # Deep Teal
    'o3 High Multimodal': '#7FC4B8',             # Muted Mint

    'GPT-4_1 Text-only': '#2B7A85',              # Deep Cyan
    'GPT-4_1 Multimodal': '#85C5D1',             # Pale Cyan
    
    'GPT-4o Text-only': '#5D4E6D',               # Deep Mauve
    'GPT-4o Multimodal': '#B5A8C4',              # Light Mauve
    
    'Gemini-2.5-Pro-High Text-only': '#A67C5B',                  # Earth Brown
    'Gemini-2.5-Pro-High Multimodal': '#F0D4B8',                 # Light Camel
    
    'DeepSeek-V3.1-Think-128k Text-only': '#A66B6B',         # Medium Rust
    'DeepSeek-V3.1-128k Text-only': '#C49494',               # Dusty Rose

    'Doubao-Seed-1-6-251015-High Text-only': '#7A6B5C',   # Deep Sandalwood

    'Kimi-K2-Thinking Text-only': '#9B8B8B',              # Lotus Silk
    
    'Qwen3-235b-A22b-Thinking-2507 Text-only': '#5C6B6B', # Celadon Gray
    
    'Intern-S1 Text-only': '#6B6B6B',                     # Ink Gray

    'ChemDFM-R Text-only': '#5B6B5B',                     # Verdigris
    
    'Ether0 Text-only': '#4A5560',                        # Indigo Slate
}


for i, (label, values) in enumerate(means.items()):
    color = color_map.get(label)
    bars = ax.bar(x + i * width, values, width, label=label, color=color)
    # Add values at top of bars
    # for j, (bar, val) in enumerate(zip(bars, values)):
    #     ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), 
    #             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metrics')
ax.set_ylabel('Mean Value')
ax.set_title('Mean Recall and Precision by Model')
ax.set_xticks(x + width * (len(labels) - 1) / 2)
ax.set_xticklabels(metrics)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 0.65)
plt.tight_layout()
plt.show()

### Ability Tag Analysis for Unmatched Nodes

In [ ]:
from collections import Counter

def count_unmatched_ability_tags(sim_dict: dict) -> Counter:
    """
    Count ability tags for all unmatched nodes in a model
    
    Args:
        sim_dict: Similarity calculation results for one model {uuid: {'node_details': {...}}}
    
    Returns:
        Counter object, containing ability_tags count for unmatched nodes
    """
    tag_counter = Counter()
    
    for uuid, result in sim_dict.items():
        node_details = result.get('node_details', {})
        for node_id, node_info in node_details.items():
            # If node is not matched
            if not node_info.get('matched', True):
                # Count all ability_tags for this node
                ability_tags = node_info.get('ability_tags', [])
                for tag in ability_tags:
                    tag_counter[tag] += 1
    
    return tag_counter

In [ ]:
def plot_unmatched_tags_per_model(sim_results: List[dict], labels: List[str], top: int = 10):
    """
    Draw a histogram for each model showing ability tags distribution of unmatched nodes
    
    Args:
        sim_results: List of similarity results for all models
        labels: Model label list
        top: Top N ability tags to display, default 10
    """
    n_models = len(sim_results)
    
    # Calculate chart layout - one subplot per row
    n_cols = 1
    n_rows = n_models
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows), sharex=True, dpi=150)
    if n_models == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for idx, (sim_dict, label) in enumerate(zip(sim_results, labels)):
        # Count ability tags for unmatched nodes
        tag_counter = count_unmatched_ability_tags(sim_dict)
        
        if not tag_counter:
            axes[idx].text(0.5, 0.5, 'No unmatched nodes', 
                          ha='center', va='center', fontsize=12)
            axes[idx].set_title(f'{label}\nNo Unmatched Nodes')
            continue
        
        # Sort by count, take top N
        sorted_tags = tag_counter.most_common(top)
        tags = [tag for tag, _ in sorted_tags]
        counts = [count for _, count in sorted_tags]
        total_count = sum(tag_counter.values())
        
        # Draw horizontal bar chart (because labels may be long)
        y_pos = np.arange(len(tags))
        bars = axes[idx].barh(y_pos, counts, alpha=0.8)
        
        # Add values at the end of bars
        for i, (bar, count) in enumerate(zip(bars, counts)):
            axes[idx].text(bar.get_width(), bar.get_y() + bar.get_height()/2,
                          f' {count}', va='center', fontsize=9)
        
        axes[idx].set_yticks(y_pos)
        axes[idx].set_yticklabels(tags, fontsize=9)
        axes[idx].set_xlabel('Count', fontsize=10)
        axes[idx].set_title(f'{label}\nTop {len(tags)} Tags (Total: {total_count})', fontsize=11)
        axes[idx].grid(axis='x', alpha=0.3)
        axes[idx].invert_yaxis()  # Highest at top
    
    # Hide extra subplots
    for idx in range(n_models, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Draw histogram of ability tags for unmatched nodes per model
plot_unmatched_tags_per_model(sim_results, labels)